# CLOY Processing

This notebook is for you to work with the CLOY data folder.

# Task 3

**Table of Content**
1. [Reading in the CLOY threads](#sec1)
2. [Aggregating all words in each thread](#sec2)
3. [Vectorizing the words of threads](#sec3)
4. [Calculating thread similarity scores](#sec4)
5. [Writing a function which identifies similar threads](#sec5)
6. [Verifying similarity using Eni's Thread Explorer app](#sec6)
7. [Removing stop words](#sec7)
8. [Advanced task](#sec8)

<h3>Import requirements</h3>

In [63]:
import json
from pprint import pprint
import string
from textblob import TextBlob
import re
from collections import Counter
import pandas as pd
import numpy as np
from numpy.linalg import norm

<h1 id="sec1">1. Reading in the CLOY threads</h1>

In [2]:
threads = []
with open(f"data/cloy_chronological.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            item = json.loads(line)
            threads.append(item)

print(f"number of submissions: {len(threads)}")

number of submissions: 3075


In [3]:
pprint(threads[0])

{'author': '[deleted]',
 'comments': [],
 'permalink': 'https://reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
 'post_id': 'dkxewd',
 'score': 1,
 'selftext': '[deleted]',
 'subreddit': 'CrashLandingOnYou',
 'timestamp': 1571646432,
 'title': 'Crash Landing On You has been created',
 'url': 'https://www.reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/'}


<h1 id="sec2">2. Aggregating all words in each thread</h1>

<h3>Define two helpful function</h3>

In [42]:
def get_nested_thread_text(thread_comments):
    text = []
    for comment in thread_comments:                                         #for each comment, save the body text
        if "comment_body" in comment:
            text.append(comment["comment_body"])
        if "replies" in comment and isinstance(comment["replies"], list):   #recursion, go until the end of each comment-reply thread
            text.extend(get_nested_thread_text(comment["replies"]))
    return text

def get_words(text):
    text = re.sub(r"[’']", "", text.lower())    #remove ' from words and lowercase everything
    text = re.sub(r"[-]", " ", text)            #replace - with space for compound words
    text = text.strip(string.punctuation)       #remove all other punctuation
    text = re.sub(r"\b(https)\S*\b", "", text)  #try to remove any https links
    return TextBlob(text).words

<h3>Iterate through each thread and extract the words</h3>

In [ ]:
data = []

for thread in threads:
    post_id = thread["post_id"]
    thread_dict = {"post_id":post_id}

    all_words_list = [thread["title"]]              #save title text
    all_words_list.append(thread["selftext"])       #save post body text

    thread_comments = thread.get("comments", [])    #save comment text
    if thread_comments:
        all_words_list.extend(get_nested_thread_text(thread_comments))

    
    full_text = " ".join(all_words_list)
    all_words_stripped = get_words(full_text)

    thread_dict["words"] = list(all_words_stripped)

    data.append(thread_dict)

In [135]:
print(len(data))

3075


In [137]:
print(threads[12]["title"])
print(threads[12]["selftext"], "\n")

print(data[12]["words"])

Check out Instagram
There’s a lot of CLOY posts on Instagram. This one is cute and timely. 
https://www.instagram.com/p/B98BicQJ0W6/?igshid=df3k5pmd1h80 

['check', 'out', 'instagram', 'theres', 'a', 'lot', 'of', 'cloy', 'posts', 'on', 'instagram', 'this', 'one', 'is', 'cute', 'and', 'timely']


<h1 id="sec3">3. Vectorizing the words of threads</h1>

<h3>Now that we have all of the words in each thread, we can vectorize the words</h4>

<h3>First we can identify all of the words across all of the threads</h3>

In [138]:
all_words = []
for thread in data:
    all_words.extend(thread["words"])

print(f"number of total words: {len(all_words)}", "\n")

all_words = set(all_words)

print(f"number of unique words: {len(all_words)}")

number of total words: 1610046 

number of unique words: 32452


<h3>Then we can count the number of instances of each word in each thread</h3>

In [139]:
thread_vectors = []
for thread in data:
    thread_words = thread["words"]
    thread_dict = {"post_id":thread["post_id"]}

    word_counts = Counter(thread_words)
    words_vector = {word: word_counts.get(word, 0) for word in all_words}

    thread_dict["words_vector"] = words_vector
    thread_vectors.append(thread_dict)

<h3>Finally, we can turn this into a dataframe like at the end of the day 11 slideshow</h3>

In [140]:
post_ids = [thread["post_id"] for thread in thread_vectors]
word_vectors = [thread["words_vector"] for thread in thread_vectors]

vectors_df = pd.DataFrame(word_vectors, index=post_ids)

In [141]:
vectors_df.head(3)

,canvas,unsolvable,floor,ect,myrwpskaprz4hhzj76nsnfucn3ay7oxj3ho3lcg3_bmfmc81co4ef4tlwx5xjkvuhz,escapade,delighting,seems,proclaiming,character.and,...,sau,plothole,reciprocal,burgeoning,scenarios,embarassment,unlikeliest,absorb,towns,cash
dkxewd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ffh52z,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
fguorr,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<h1 id="sec4">4. Calculating thread similarity scores</h1>

<h3>Define two helpful functions, one from the day 11 slides</h3>

In [142]:
def cosineSimilarity(vec1, vec2):
    V1 = np.array(vec1)
    V2 = np.array(vec2)
    cosine = np.dot(V1, V2)/(norm(V1)*norm(V2))
    return cosine

def compare_two_threads(post_id_one, post_id_two):
    post_ids = vectors_df.index.tolist()
    if post_id_one not in post_ids or post_id_two not in post_ids:
        print("at least one provided post_id is not in the dataframe.")
        return
    else:
        V1 = vectors_df.loc[post_id_one].tolist()
        V2 = vectors_df.loc[post_id_two].tolist()
        cosine_similarity = cosineSimilarity(V1, V2)
        return float(cosine_similarity)

<h3>Test using a fake post_id</h3>

In [143]:
compare_two_threads("fzk9n3", "notapostid")

at least one provided post_id is not in the dataframe.


<h3>Test using real post_ids</h3>

In [144]:
print("both threads about sad emotional reactions to the show")
cs = compare_two_threads("fzk9n3", "g3kmhf")
print(f"these threads have a cosine similarity of: {cs}", "\n")

print("one thread about sad emotion reactions and one about plot holes")
cs = compare_two_threads("fzk9n3", "ggy0tg")
print(f"these threads have a cosine similarity of: {cs}", "\n")

both threads about sad emotional reactions to the show
these threads have a cosine similarity of: 0.501146374129895 

one thread about sad emotion reactions and one about plot holes
these threads have a cosine similarity of: 0.3503655735193813 



<h1 id="sec5">5. Writing a function which identifies similar threads</h1>

<h3>Write the function</h3>

In [153]:
def get_most_similar_threads(input_post_id):
    """Returns the top 5 most similar threads based on cosine similarity"""
    post_ids = vectors_df.index.tolist()
    post_ids = [post_id for post_id in post_ids if post_id != input_post_id]

    cosine_similarity_dict = {}
    for post_id in post_ids:
        cosine_similarity_dict[post_id] = compare_two_threads(input_post_id, post_id)

    top_three = sorted(cosine_similarity_dict, key=cosine_similarity_dict.get, reverse=True)[:5]
    top_three = {
        top_three[0]: cosine_similarity_dict[top_three[0]],
        top_three[1]: cosine_similarity_dict[top_three[1]],
        top_three[2]: cosine_similarity_dict[top_three[2]],
        top_three[3]: cosine_similarity_dict[top_three[3]],
        top_three[4]: cosine_similarity_dict[top_three[4]]
    }
    return top_three

<h3>Define a function for printing information</h3>

In [156]:
def print_similar_thread_info(input_post_id):
    post_ids = vectors_df.index.tolist()

    top_five_similar = get_most_similar_threads(input_post_id)\

    og_index = post_ids.index(input_post_id)
    og_title = threads[og_index].get("title")
    print(f"{input_post_id} | {og_title}", "\n")

    count = 1
    for post in top_five_similar:
        post_index = post_ids.index(post)
        post_title = threads[post_index].get("title")
        print(f"{count}. {top_five_similar[post]}")
        print(f"{post} | {post_title}", "\n")
        count += 1
    
    return top_five_similar

<h3>Test the function</h3>

<h3>Input post is about Christmas</h3>

In [157]:
print_similar_thread_info("kjm5rm")
print()

kjm5rm | It’s Christmas, so here’s a Christmas question 

1. 0.6485245402514568
1gxh5w5 | The 2 weeks of the year 

2. 0.6478080906340542
1iglz4k | Was it a pun? 

3. 0.6404732026974536
hmk21m | Christmas card??? 

4. 0.6400889737628791
iwira6 | Underrated moment #20: the Christmas tree 

5. 0.6179211410108363
15apd9p | Do you think HB feels pressure to live up to RJH? 




<h3>Input post is about Pyongyang, the capital in North Korea</h3>

In [158]:
print_similar_thread_info("1973gmh")
print()

1973gmh | Article about Pyongyang 

1. 0.8350171670054688
oh2raz | Interesting article about recent purge in North Korea... 

2. 0.8282026729785784
1gu4oas | Send help! Just finished watching 😭 

3. 0.8224895008226002
ridyxj | Article about current situation in NK 

4. 0.8207795089161602
1de6px3 | Review of “Crash Landing on You” overrated and overhyped!  

5. 0.8193524450047761
1ebb95p | Was CLOY a social commentary on Koreas division and potential for reunification? 




<h1 id="sec6">6. Verifying similarity using Eni's Thread Explorer app</h1>

<h3>We can see that these threads are similar by using the printing method which shows the titles, but looking at the Thread Explorer app, we can look at the comments and confirm that they talk about similar things (for these examples, atleast!). I definitely tried to pick out input threads that are about a specific thing that wouldn't be a theme across most of the posts (for example, like about the character ship), so that might take it easier to just tell from a glance that they are infact similar.</h3>

<h1 id="sec7">7. Removing stop words</h1>

<h3>We can rewrite some code to test how getting rid of the stop words affects these test results</h3>

<h3>I googled about TextBlob and stop words, and it recommends to use the nltk library like this</h3>

In [160]:
import nltk
from nltk.corpus import stopwords

In [161]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chick\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


<h3>First, let's rewrite the "get_words" function which cleans all of the text for a thread</h3>

In [162]:
def get_words(text):
    text = re.sub(r"[’']", "", text.lower())    #remove ' from words and lowercase everything
    text = re.sub(r"[-]", " ", text)            #replace - with space for compound words
    text = text.strip(string.punctuation)       #remove all other punctuation
    text = re.sub(r"\b(https)\S*\b", "", text)  #try to remove any https links
    text = TextBlob(text).words
    text = [word for word in text if word not in stop_words]
    return text

<h3>Now, we can rerun the code which builds the dataframe</h3>

In [163]:
data = []

for thread in threads:
    post_id = thread["post_id"]
    thread_dict = {"post_id":post_id}

    all_words_list = [thread["title"]]              #save title text
    all_words_list.append(thread["selftext"])       #save post body text

    thread_comments = thread.get("comments", [])    #save comment text
    if thread_comments:
        all_words_list.extend(get_nested_thread_text(thread_comments))

    
    full_text = " ".join(all_words_list)
    all_words_stripped = get_words(full_text)

    thread_dict["words"] = list(all_words_stripped)

    data.append(thread_dict)

In [164]:
print(threads[12]["title"])
print(threads[12]["selftext"], "\n")

print(data[12]["words"])

Check out Instagram
There’s a lot of CLOY posts on Instagram. This one is cute and timely. 
https://www.instagram.com/p/B98BicQJ0W6/?igshid=df3k5pmd1h80 

['check', 'instagram', 'theres', 'lot', 'cloy', 'posts', 'instagram', 'one', 'cute', 'timely']


<p>We can see that the 'unimportant' words get removed</p>

In [165]:
all_words = []
for thread in data:
    all_words.extend(thread["words"])

print(f"number of total words: {len(all_words)}", "\n")

all_words = set(all_words)

print(f"number of unique words: {len(all_words)}")

number of total words: 835423 

number of unique words: 32312


<p>Before removing the stop words, we had: 

number of total words: 1610046 

number of unique words: 32452

So, we can see that this number has reduced a lot of the total words, but there are still a similar amount of unique words</p>

In [166]:
thread_vectors = []
for thread in data:
    thread_words = thread["words"]
    thread_dict = {"post_id":thread["post_id"]}

    word_counts = Counter(thread_words)
    words_vector = {word: word_counts.get(word, 0) for word in all_words}

    thread_dict["words_vector"] = words_vector
    thread_vectors.append(thread_dict)

In [167]:
post_ids = [thread["post_id"] for thread in thread_vectors]
word_vectors = [thread["words_vector"] for thread in thread_vectors]

vectors_df = pd.DataFrame(word_vectors, index=post_ids)

<h3>Now, we can check our tests about Christmas and Pyongyang again</h3>

In [172]:
print_similar_thread_info("kjm5rm")
print()

C:\Users\chick\AppData\Local\Temp\ipykernel_18368\2686249190.py:4: RuntimeWarning: invalid value encountered in scalar divide
  cosine = np.dot(V1, V2)/(norm(V1)*norm(V2))


kjm5rm | It’s Christmas, so here’s a Christmas question 

1. 0.5855491813389548
hmk21m | Christmas card??? 

2. 0.48443029790657854
1iglz4k | Was it a pun? 

3. 0.46966122041347336
1gxh5w5 | The 2 weeks of the year 

4. 0.469453272638125
iwira6 | Underrated moment #20: the Christmas tree 

5. 0.4658709159295794
112ikga | I asked ChatGPT to come up with CLOY continuation scripts and here are 3 of them. 




In [173]:
print_similar_thread_info("1973gmh")
print()

C:\Users\chick\AppData\Local\Temp\ipykernel_18368\2686249190.py:4: RuntimeWarning: invalid value encountered in scalar divide
  cosine = np.dot(V1, V2)/(norm(V1)*norm(V2))


1973gmh | Article about Pyongyang 

1. 0.35620011144982766
oh2raz | Interesting article about recent purge in North Korea... 

2. 0.3447146304290885
ridyxj | Article about current situation in NK 

3. 0.34395640371706
kpu73v | My thoughts as this was my first kdrama 

4. 0.33700309884896584
n74uhk | I have questions about Korean culture.... 

5. 0.3358633422559938
iufpmn | CLOY set my expectations to high as my first kdrama 




<h3>We can see from the examples that when we remove stop words, some of the threads which were previously in the top 5 most similar aren't anymore, and the cosine similarities between all threads drops. For the Christmas threads from ~0.6 to ~0.5 and for the Pyongyang threads from ~0.8 to ~0.3. I'm curious as to why the Christmas threads didn't see as large of a drop in cosine similarity, perhaps the Pyongyang threads contained more stop words? Not sure!</h3>

<h1 id="sec8">8. Advanced task</h1>

<h3>Unfortunately I don't have time this week to work on the advanced task as I am already turning the assignment in late, but I will definitely try to work on it independently once I have more time!</h3>